In [ ]:
using LinearAlgebra, MPSDynamics, Statistics, FFTW, StaticArrays
include("/Users/travail/Desktop/Stage_Quantic/Code_Julia/coupled_harmonic_oscillators_solver.jl")
include("/Users/travail/Desktop/Stage_Quantic/Code_Julia/Plot_setting.jl")

# Modelisation of the Caldeira-Leggett model in the linear case

We rely on the article "Thorbeck, Ted, Alexander McDonald, O. Lanes, John Blair, George Keefe, Adam A. Stabile, Baptiste Royer, Luke C. G. Govia, et Alexandre Blais. « High-Fidelity Gates in a Transmon Using Bath Engineering for Passive Leakage Reset ». arXiv, 6 novembre 2024. https://doi.org/10.48550/arXiv.2411.04101." The bath is colored via the purcell filter, creating a pass-band filter. The frequency absorption for the bath is between $\omega_{min}$ and $\omega_{max}$. We will take $\Re(Z)=Z_0$ in the window and 0 outside, with a spooth step. The real part is non zero on a 2 GHz range.

For a Caldeira-Leggett approach: $\omega_m=\omega_{min}+m\frac{\omega_{max}-\omega_{min}}{n}$, $C_m=\frac{1}{\Re(Z)(\omega_m)\omega_m}$.

For the oscillator, we set the frequency in the middle of the window $\omega_0=3.5$ GHz and we choose the same capacitance order of magnitude as the one given for the mode $b$ in Lescanne, Raphaël, Marius Villiers, Théau Peronnin, Alain Sarlette, Matthieu Delbecq, Benjamin Huard, Takis Kontos, Mazyar Mirrahimi, et Zaki Leghtas. « Exponential Suppression of Bit-Flips in a Qubit Encoded in an Oscillator ». Nature Physics 16, nᵒ 5 (mai 2020): 509‑13. https://doi.org/10.1038/s41567-020-0824-x, so $E_C=73.5$ MHz and

\begin{equation}
    1/C=2E_C/e^2=2\times2\pi\frac{E_C}{h}\frac{\hbar}{e^2}=0.147\times2\pi GHz,
\end{equation}
where we set $\hbar=1$ and $e$=1.


It will be easier to work with $1/C$ and $1/C_m$.

Then, following Jeffrey, Evan, Daniel Sank, J. Y. Mutus, T. C. White, J. Kelly, R. Barends, Y. Chen, et al. « Fast Accurate State Measurement with Superconducting Qubits ». Physical Review Letters 112, nᵒ 19 (15 mai 2014): 190504. https://doi.org/10.1103/PhysRevLett.112.190504, we set the dissipation rate of the bath to $\kappa=1/20$ ns and following the lecture notes of Michel Devoret,
\begin{equation}
    Z_0=\frac{1}{2\kappa C}.
\end{equation}

The eigenfrequencie of the buffer mode is $\omega_b/2\pi\approx 5$ GHz, so that we take $\omega_{min}/2\pi=4$ GHz and $\omega_{max}/2\pi=6$ GHz.

We start by working at zero temperature

In [ ]:
function smoothstep(x)
    return x^3 * (x * (x * 6 - 15) + 10)
end

In [ ]:
function FFT_perso(ϕ,t)
    F = fftshift(fft(ϕ))
    fs = length(t)/(maximum(t)-minimum(t)) # sampling frequency
    freqs = fftshift(fftfreq(length(t), fs))
    return F, freqs
end

function comparison_simu(simu_list,E, E_params)
    ϕ_list = []
    q_list = []
    t_list = []
    E_list = []
    spectrum_list = []

    for simu in simu_list
        ϕ, q, t = simu
        push!(ϕ_list, ϕ)
        push!(q_list, q)
        push!(t_list, t)
        push!(E_list, E(ϕ, q, E_params))
        spectrum = FFT_perso(ϕ, t)
        push!(spectrum_list, spectrum)
    end

    return(E_list, spectrum_list)
end

function Energy_linear(ϕ, q, (L, C_inv))
    """
    INPUT :
    ϕ : vector of the field ϕ
    q : vector of the field q
    L : inductance
    C_inv : inverse capacitance in GHz

    OUTPUT :
    Energy of the linear system
    """
    return q.^2 /2 *C_inv + ϕ.^2 / (2*L)
end;

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
n = 500 # length of the chain

#### Caracterization of the LC circuit #####
ω_sys = 5*2*pi # GHz
C_sys_inv = 0.147*2*pi #ħ/e^2 GHz from Lescanne et al.
L_sys=C_sys_inv/ω_sys^2 #ħ/e^2 /GHz
Z_sys = C_sys_inv/ω_sys #ħ/e^2
κ=1/20 # 1/ns from Jeffrey et al. 

ωc_up=10*2*pi # Upper cut-off of the spectral density J(ω), GHz
ωc_dwn=0.0*2*pi # Lower cut-off of the spectral density J(ω), GHz

function Ohmic_bath(n;ωc_dwn=0.0*2*pi, ωc_up=10*2*pi)
    """
    INPUT : 
    n : number of oscillators in the chain
    ω_min : lower cut-off frequency of the spectral density J(ω) in GHz
    ω_max : upper cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    onsite_energies : vector of on-site energies ϵ_i of the chain
    hopping_energies : vector of hopping energies t_i of the chain
    system_chain_coupling : coupling between the system and the chain
    """
    ### Caracterization of the impedance #####
    α = 2*κ/C_sys_inv # no unit

    # We design the impedance to be a smooth pass-band filter
    
    ω_min = .2 * 2 * pi# 3.886 *2*pi # GHz

    ω_max = ωc_up-.2*2*pi # Lower cut-off of the spectral density J(ω), GHz
    ω_min = ωc_dwn+.2*2*pi # Cut-off of the spectral density J(ω), GHz


    # AB segments correspond to the different frequency ranges where the spectral density is defined.
    # It corresponds to i=1 ; i=2 and i=3 of the Jω_fct(ω,i) function.

    AB = [[-Inf ωc_dwn];[ωc_dwn ω_min];[ω_min ω_max];[ω_max ωc_up];[ωc_up Inf]]
    #AB=[[-Inf ω_min];[ω_min ω_max];[ω_max Inf]]


    function Jω_fct(ω,i)
        if i==1 # First segment of AB (here [-Inf ω_min]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [ω_min ω_max])
            y = @. smoothstep((ω - ωc_dwn) / (ω_min - ωc_dwn))
        elseif i==3 # Third segment of AB (here [ω_min ω_max])
            y = 1
        elseif i==4 # Fourth segment of AB (here [ω_max ωc_up])``
            y = @. (1 - smoothstep((ω - ω_max) / (ωc_up - ω_max)))
        elseif i==5 # Fifth segment of AB (here [ωc_up Inf])
            y = 0
        end
        return y*α.*ω
    end

    onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=Jω_fct, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

    return onsite_energies, hopping_energies, system_chain_coupling
end;

In [ ]:
function endsiteocc_2(t, cparams)
    es=cparams[1]
    ts=cparams[2]
    N=length(es)
    hmat = diagm(0=>es, 1=>ts, -1=>ts)
    F = eigen(hmat)
    U = F.vectors
    S = F.values
    real.(transpose(U[:,1].*exp.(im*t.*S))*U[:,N]*transpose(U[:,N])*(U[:,1].*exp.(-im*t.*S)))
end
function findchainlength_2(T, cparams::Tuple{Vector{Float64}, Vector{Float64}, Float64}; eps=10^-4, verbose=false)
    Nmax = length(cparams[1])
    occprev = endsiteocc_2(T, [cparams[1][1:Nmax], cparams[2][1:Nmax-1]])
    occ = endsiteocc_2(T, [cparams[1][1:Nmax-1], cparams[2][1:Nmax-2]])

    verbose && println("truncating chain...")
    verbose && println("starting chain length $Nmax")

    if abs(occ-occprev) > eps
        throw(error("Suitable truncation not found, try larger starting chain length"))
    end
    occprev=occ
    for ntrunc=Nmax-2:-1:1
        verbose && println("ntrunc = $ntrunc")
        occ = endsiteocc_2(T, [cparams[1][1:ntrunc], cparams[2][1:ntrunc-1]])
        if abs(occ-occprev) > eps
            return ntrunc
        end
        occprev=occ
    end
end

In [ ]:
n_huge=1000
params = Ohmic_bath(n_huge,ωc_up=30*2*pi)
t_max = 1/κ
n_opt = findchainlength_2(t_max, params; eps=10^-4, verbose=false);

In [ ]:
onsite_energies,hopping_energies,system_chain_coupling=Ohmic_bath(n_opt+10);

In [ ]:
p1=scatter(onsite_energies, markersize=1)
p2=scatter(hopping_energies, markersize=1)
plot(p1,p2,layout=(1,2),legend=:topright)
a=ωc_dwn
b=ωc_up
hline!([(a+b)/2 (b-a)/4])

In [ ]:
### Caldeira Leggett model ###

L_sys_prime_inv = 1/L_sys #+ 2/3*Z*(ω_max^(3/2)-ω_min^(3/2))
ω_sys_prime = sqrt((C_sys_inv/L_sys)) # GHz
Z_sys_prime = sqrt(L_sys*C_sys_inv)

### Mapping ###
ω = vcat(ω_sys_prime,onsite_energies) # GHz
g_q=vcat(0, -hopping_energies) 
g_ϕ=vcat(-system_chain_coupling*sqrt(Z_sys_prime), hopping_energies) 

ϕ_init = 1.0
q_init = 1.0

ϕ_0 = vcat(SVector(ϕ_init/sqrt(Z_sys_prime)),SVector{n}(zeros(n)))
q_0 = vcat(SVector(q_init*sqrt(Z_sys_prime)),SVector{n}(zeros(n)))

Mq=SymTridiagonal(ω, g_q)
Mϕ=SymTridiagonal(ω, g_ϕ)

A=Mq * Mϕ
vals, vecs = eigen(A)



In [ ]:





ωk² = real(vals)
inv_vecs = inv(vecs)
scatter(ωk²[1:end], label="Eigenvalues",markersize=1, markeralpha=.1)



#t, ϕ_t, q_t,E = solve_normal_modes(ω, g, C, ϕ_0, q_0, tspan, energy=true)
#p1=plot(sol; legend=:outertopright)
#p2=plot(compute_total_energy(sol,ω, g, C, N))
#plot(p1,p2,layout=(1,2), size=(1200,400))

In [ ]:
ωk = sqrt.(ωk²)
P = real(vecs)  # columns = normal modes

P_inv=inv(P)

# Project initial conditions into normal mode basis
a0 = (P_inv * ϕ_0)
adot0 = (P_inv * Mq * q_0)

tspan = (0.0, t_max)
dt=1/(20*maximum(ωk))
npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points


# Time evolution in normal coordinates
tlist = range(tspan[1], tspan[2], length=npoints)
a_t = [a0 .* cos.(ωk .* t) .+ (adot0 ./ ωk) .* sin.(ωk .* t) for t in tlist]
a_t_dot= [a0 .* (-ωk) .* sin.(ωk .* t) .+ (adot0) .* cos.(ωk .* t) for t in tlist]

# Reconstruct real-space fields
ϕ_t = [P * a for a in a_t]*sqrt(Z_sys_prime)
inv_Mq= inv(Mq)
q_t = [inv_Mq * P * a for a in a_t_dot] / sqrt(Z_sys_prime)  # q = \dot{ϕ}/Mq

ϕ_sys_t =  [ϕ_t[i][1] for i in 1:length(tlist)]
q_sys_t =  [q_t[i][1] for i in 1:length(tlist)];


In [ ]:
p1=plot(tlist*κ, ϕ_sys_t, label="ϕ(t)")
p2=plot(tlist*κ, q_sys_t, label="q(t)")
plot(p1,p2,layout=(1,2),legend=:topright, xlabel=["κt" "κt"])
#savefig("test.pdf")

In [ ]:
plot(ϕ_sys_t, q_sys_t, label="Ohmic bath", xlabel="ϕ", ylabel="q", legend=:topright, linewidth=.2)

In [ ]:
E = 0.5 * [q_t[t]' * (Mq*Z_sys_prime) * q_t[t] + ϕ_t[t]' * (Mϕ/Z_sys_prime) * ϕ_t[t] for t in 1:npoints]
plot(tlist,E, label="E(t)", xlabel="t")

In [ ]:
X_0=[ϕ_init,q_init]
t=tlist
Ω=sqrt(ω_sys^2-κ^2)
q_analytic= @. exp(-κ*t)*(X_0[2]*cos(Ω*t) - (κ*X_0[2]+ω_sys^2*X_0[1]/C_sys_inv)*sin(Ω*t)/Ω)
ϕ_analytic= @. exp(-κ*t)*(X_0[1]*cos(Ω*t) + (X_0[2]*C_sys_inv+κ*X_0[1])*sin(Ω*t)/Ω);


ls=-2κ/C_sys_inv*(ωc_up+ω_sys*log(ωc_up/ω_sys-1))/pi
Ω_ls=sqrt((ω_sys+ls)^2-κ^2)
q_analytic_ls= @. exp(-κ*t)*(X_0[2]*cos((Ω_ls)*t) - (κ*X_0[2]+ω_sys^2*X_0[1]/C_sys_inv)*sin((Ω_ls)*t)/Ω)
ϕ_analytic_ls= @. exp(-κ*t)*(X_0[1]*cos((Ω_ls)*t) + (X_0[2]*C_sys_inv+κ*X_0[1])*sin((Ω_ls)*t)/Ω);

q_analytic_ls= @. exp(-κ*t)*(X_0[2]*cos(ω_sys*t) - (κ*X_0[2]+ω_sys^2*X_0[1]/C_sys_inv)*sin(ω_sys*t)/Ω)
ϕ_analytic_ls= @. exp(-κ*t)*(X_0[1]*cos(ω_sys*t) + (X_0[2]*C_sys_inv+κ*X_0[1])*sin(ω_sys*t)/Ω);

In [ ]:
ωc_up

In [ ]:
(2κ/C_sys_inv*(30*2pi-ω_sys*log(30*2pi/ω_sys-1)))*Z_sys_prime/sqrt(2)

In [ ]:
(2κ/C_sys_inv*(30*2pi))*Z_sys_prime/2/0.2997354139770394

In [ ]:
(2κ/C_sys_inv*(10*2pi))*Z_sys_prime/sqrt(2)

In [ ]:
(2κ/C_sys_inv*(10*2pi))/0.14494355389807723

In [ ]:
plot(tlist*κ, ϕ_sys_t, label="ϕ₀(t) 1d-chain", xlabel="t", xlims=(0,.1))
plot!(t*κ, ϕ_analytic, label="ϕ₀(t) CL master equation", xlabel="κt")
title!("Comparison of the 1D chain and the CL master \n equation for a trapezoidal spectral density")
#savefig("1D_chain_vs_CL_trapezoid.pdf")

In [ ]:
plot(ϕ_sys_t[1:1000], q_sys_t[1:1000], label="Ohmic bath", linewidth=.2, color = "blue")
plot!(ϕ_analytic[1:1000], q_analytic[1:1000], label="Analytic", xlabel="ϕ", ylabel="q", legend=:topright, linewidth=.2, color = "red")
plot!(xlabel="ϕ", ylabel="q", legend=:topright)

In [ ]:
F = fftshift(fft([ϕ_t[i][1] for i in 1:length(tlist)]))
fs = length(tlist)/(tspan[2]-tspan[1]) # sampling frequency
freqs = fftshift(fftfreq(length(tlist), fs))

# plots 
time_domain = plot(tlist*κ, [ϕ_t[i][1] for i in 1:length(tlist)], title = "Signal", label='ϕ')
freq_domain = plot(freqs, abs.(F), title = "Spectrum", xlim=(4.8, 5.2), label="abs.(F)") 
f_max=abs(freqs[argmax(real(F))])
vline!(freq_domain, [f_max], label = "ω₀")
plot(time_domain, freq_domain, layout = (2,1))

#savefig("FFT_trapezoid_cut_off_30.pdf")

In [ ]:
argmax(P[1, :])

In [ ]:
ω_sys -  ωk[argmax(P[1, :])]

In [ ]:
f_max

In [ ]:
ω_sys/2pi-κ/3

In [ ]:
plot(tlist*κ, ϕ_analytic, label="ϕ₀(t) 1d-chain", xlabel="t", xlims=(0,.1))
plot!(t*κ, ϕ_analytic_l, label="ϕ₀(t) CL master equation", xlabel="κt")
title!("Comparison of the 1D chain and the CL master \n equation for a trapezoidal spectral density")
#savefig("1D_chain_vs_CL_trapezoid.pdf")

In [ ]:
plot(tlist*κ, [ϕ_t[i][1] for i in 1:length(tlist)], label="ϕ₀(t) 1d-chain", xlabel="t", xlims=(0,3))
plot!(t*κ, ϕ_analytic, label="ϕ₀(t) CL master equation", xlabel="κt")
title!("Comparison of the 1D chain and the CL master \n equation for a trapezoidal spectral density")
#savefig("1D_chain_vs_CL_trapezoid_long.pdf")

In [ ]:
plot(tlist, [q_t[i][1] for i in 1:length(tlist)], label="q₀(t) 1d-chain", xlabel="t")
plot!(t, q_analytic, label="q₀(t) CL master equation", xlabel="t [ns]")

In [ ]:
plot(tlist*κ, [ϕ_t[i][1] for i in 1:length(tlist)], label="ϕ₀(t) 1d-chain", xlabel="t", xlim=(0,.2))
plot!(t*κ,ϕ_analytic, label="ϕ₀(t) CL master equation", xlabel="κt")
savefig("1D_chain_vs_CL_trapezoid_short.pdf")

In [ ]:
av_ϕ_analitic=averaging(ϕ_analytic.^2,80)
av_ϕ_t_0=averaging([ϕ_t[i][1] for i in 1:length(tlist)].^2,80)
plot(abs.(av_ϕ_t_0-av_ϕ_analitic)./av_ϕ_analitic,yaxis=:log)#, xlabel="κt", xlims=(0,1),yaxis=:log, ylims=(1e-10,1e4))
title!("Normalized error of the 1D chain \n and the CL master equation for a trapezoidal spectral density")

## Exponential decay for the tail of the Spectral Function

In [ ]:
function exp_decay(n ; ω_c=50*2pi, ω_cut_off=90, plot_bool= false)
    """
    INPUT :
    n : number of oscillators in the chain
    ω_c : characteristic frequency of the exponential decay in GHz
    ω_cut_off : cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    onsite_energies : vector of on-site energies ϵ_i of the chain
    hopping_energies : vector of hopping energies t_i of the chain
    system_chain_coupling : coupling between the system and the chain
    """
    
    ### Caracterization of the impedance #####
    Y = (C_sys_inv/(2*κ))^(-1) #ħ/e^2

    AB_exp_decay=[[-Inf 0];[0 ω_cut_off];[ω_cut_off Inf]]

    function J_exp_decay(ω,i)
        if i==1 # First segment of AB (here [-Inf 0]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [0 ω_cut_off])
            y = @.Y*ω*exp(-ω/ω_c)
        elseif i==3 # Third segment of AB (here [ω_cut_off Inf])
            y=0
        end
        return y
    end
    if plot_bool
        ω = range(0, stop=ω_cut_off, length=1000)
        J_values = [J_exp_decay(ω[i], 2) for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    else
        onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J_exp_decay, mc=size(AB_exp_decay)[1], mp=0, AB=AB_exp_decay, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
        return onsite_energies, hopping_energies, system_chain_coupling
    end
end;

In [ ]:
onsite_energies,hopping_energies,system_chain_coupling = exp_decay(n)

In [ ]:
p1=scatter(onsite_energies)
p2=scatter(hopping_energies)
plot(p1,p2,layout=(1,2),size=(1200,400),legend=:topright)

In [ ]:
### Caldeira Leggett model ###

L_sys_prime_inv = 1/L_sys #+ 2/3*Z*(ω_max^(3/2)-ω_min^(3/2))
ω_sys_prime = sqrt((C_sys_inv/L_sys)) # GHz
Z_sys_prime = sqrt(L_sys*C_sys_inv)

### Mapping ###
ω = vcat(ω_sys_prime,onsite_energies) # GHz
g_q=vcat(0, -hopping_energies) 
g_ϕ=vcat(-system_chain_coupling*sqrt(Z_sys_prime), hopping_energies) 

ϕ_init = 1.0
q_init = 1.0

ϕ_0 = vcat(SVector(ϕ_init/sqrt(Z_sys_prime)),SVector{n}(zeros(n)))
q_0 = vcat(SVector(q_init*sqrt(Z_sys_prime)),SVector{n}(zeros(n)))

Mq=SymTridiagonal(ω, g_q)
Mϕ=SymTridiagonal(ω, g_ϕ)

A=Mq * Mϕ
vals, vecs = eigen(A)



In [ ]:
ωk² = (vals)
scatter(ωk²[1:end], label="Eigenvalues", yaxis=:log)

In [ ]:
ωk = sqrt.(ωk²)
P = real(vecs)  # columns = normal modes

P_inv=inv(P)

# Project initial conditions into normal mode basis
a0 = (P_inv * ϕ_0)
adot0 = (P_inv * Mq * q_0)

tspan = (0.0, 1/κ)#max(20π / minimum(ωk), 1/κ))
dt=1/(10*maximum(ωk))
npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points


# Time evolution in normal coordinates
#tlist = range(tspan[1], tspan[2], length=npoints)
a_t = [a0 .* cos.(ωk .* t) .+ (adot0 ./ ωk) .* sin.(ωk .* t) for t in tlist]
a_t_dot= [a0 .* (-ωk) .* sin.(ωk .* t) .+ (adot0) .* cos.(ωk .* t) for t in tlist]

# Reconstruct real-space fields
ϕ_e = [P * a for a in a_t]*sqrt(Z_sys_prime)
inv_Mq= inv(Mq)
q_e = [inv_Mq * P * a for a in a_t_dot] / sqrt(Z_sys_prime)  # q = \dot{ϕ}/Mq



In [ ]:
plot(tlist, [ϕ_e[i][1] for i in 1:length(tlist)], label="ϕ₀(t) symplectic", xlabel="t", xlim=(0,40))
plot!(tlist, [q_e[i][1] for i in 1:length(tlist)], label="q₀(t) symplectic", xlabel="t")
title!("Spectral function with exponential decay")

In [ ]:
plot(tlist*κ, [ϕ_e[i][1] for i in 1:length(tlist)], label="ϕ₀(t) 1d-chain", xlabel="t", xlim=(0,1))
plot!(t*κ, ϕ_analytic, label="ϕ₀(t) CL master equation", xlabel="κt")
title!("Comparison of the 1D chain and the CL master equation \n for an ohmic spectral density with an exponential decay")
#savefig("1D_chain_vs_CL_exponential.pdf")

In [ ]:
F = fftshift(fft([ϕ_e[i][1] for i in 1:length(tlist)]))
fs = length(tlist)/(maximum(tlist)-minimum(tlist)) # sampling frequency
freqs = fftshift(fftfreq(length(tlist), fs))

# plots 
time_domain = plot(tlist, [ϕ_e[i][1] for i in 1:length(tlist)], title = "Signal", label='f')
freq_domain = plot(freqs, abs.(F), title = "Spectrum", xlim=(4.8, 5.2), label="abs.(F)") 
f_max=abs(freqs[argmax(abs.(F))])
vline!(freq_domain, [f_max], label = "ω₀")
plot(time_domain, freq_domain, layout = (2,1))

In [ ]:
f_max

In [ ]:
ω_sys/2pi-2*κ/3

In [ ]:
av_ϕ_e_0=averaging([ϕ_e[i][1] for i in 1:length(tlist)].^2,80)
plot(abs.(av_ϕ_e_0-av_ϕ_analitic)./av_ϕ_analitic, xlabel="κt", xlims=(0,300),yaxis=:log)
title!("Normalized error of the 1D chain \n and the CL master equation for a trapezoidal spectral density")

## Drude-Lorentz Spectral Function

In [ ]:
function drude_lorentz(n ; Ω=50*2pi, ω_cut_off=100, plot_bool= false)
    """
    INPUT :
    n : number of oscillators in the chain
    Ω : characteristic frequency of the Drude-Lorentz function in GHz
    ω_cut_off : cut-off frequency of the spectral density J(ω) in GHz

    OUTPUT :
    onsite_energies : vector of on-site energies ϵ_i of the chain
    hopping_energies : vector of hopping energies t_i of the chain
    system_chain_coupling : coupling between the system and the chain
    """

    AB_drude= [[-Inf 0];[0 ω_cut_off];[ω_cut_off Inf]]
    function J_drude(ω,i)
        if i==1 # First segment of AB (here [-Inf 0]), zero for negative frequencies when β == Inf
            y = 0
        elseif i==2 # Second segment of AB (here [0 ω_cut_off])
            y = @. 2*κ/(C_sys_inv)*ω*Ω^2/((ω^2+Ω^2))
        elseif i==3 # Third segment of AB (here [ω_cut_off Inf])
            y = 0
        end
        return y
    end
    if plot_bool
        ω = range(0, stop=ω_cut_off, length=1000)
        J_values = [J_drude(ω[i], 2) for i in 1:length(ω)]
        plot(ω, J_values, label="J(ω)", xlabel="ω (GHz)", ylabel="J(ω)")
    else
        onsite_energies,hopping_energies,system_chain_coupling = chaincoeffs_finiteT(n, β, false; J=J_drude, mc=size(AB_drude)[1], mp=0, AB=AB_drude, iq=1, idelta=2, procedure=:Lanczos, Mmax=5000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0
        return onsite_energies, hopping_energies, system_chain_coupling
    end
end;

In [ ]:
n=500
onsite_energies,hopping_energies,system_chain_coupling=drude_lorentz(n, ω_cut_off=250)

In [ ]:
p1=scatter(onsite_energies)
p2=scatter(hopping_energies)
plot(p1,p2,layout=(1,2),size=(1200,400),legend=:topright)

In [ ]:
### Caldeira Leggett model ###

L_sys_prime_inv = 1/L_sys #+ 2/3*Z*(ω_max^(3/2)-ω_min^(3/2))
ω_sys_prime = sqrt((C_sys_inv/L_sys)) # GHz
Z_sys_prime = sqrt(L_sys*C_sys_inv)

### Mapping ###
ω = vcat(ω_sys_prime,onsite_energies) # GHz
g_q=vcat(0, -hopping_energies) 
g_ϕ=vcat(-system_chain_coupling*sqrt(Z_sys_prime), hopping_energies) 

ϕ_init = 1.0
q_init = 1.0

ϕ_0 = vcat(SVector(ϕ_init/sqrt(Z_sys_prime)),SVector{n}(zeros(n)))
q_0 = vcat(SVector(q_init*sqrt(Z_sys_prime)),SVector{n}(zeros(n)))

Mq=SymTridiagonal(ω, g_q)
Mϕ=SymTridiagonal(ω, g_ϕ)

A=Mq * Mϕ
vals, vecs = eigen(A)



In [ ]:
ωk² = (vals)
scatter(ωk²[1:end], label="Eigenvalues")

In [ ]:
down_lim,up_lim=1,length(vals)
ωk = sqrt.(vals[down_lim:up_lim])
P = vecs  # columns = normal modes
P_cut = vecs[down_lim:up_lim, down_lim:up_lim]  # cut the first row and column
P_inv=inv(P)
P_inv_cut = P_inv[down_lim:up_lim, down_lim:up_lim]  # cut the first row and column
# Project initial conditions into normal mode basis
a0 = (P_inv * ϕ_0)[down_lim:up_lim]
adot0 = (P_inv * Mq * q_0)[down_lim:up_lim]

tspan = (0.0, 1/κ)#max(20π / minimum(ωk), 1/κ))
dt=1/(10*maximum(ωk))
npoints = Int(round((tspan[2]-tspan[1])/dt))  # number of time points




In [ ]:
# Time evolution in normal coordinates
#tlist = range(tspan[1], tspan[2], length=npoints)
a_t = [a0 .* cos.(ωk .* t) .+ (adot0 ./ ωk) .* sin.(ωk .* t) for t in tlist]
a_t_dot= [a0 .* (-ωk) .* sin.(ωk .* t) .+ (adot0) .* cos.(ωk .* t) for t in tlist]

# Reconstruct real-space fields
ϕ_l = [P_cut * a for a in a_t] *sqrt(Z_sys_prime)
inv_Mq= inv(Mq)
inv_Mq_cut =(P_inv * inv_Mq * P)[down_lim:up_lim, down_lim:up_lim] / sqrt(Z_sys_prime)
q_l = [P_cut * inv_Mq_cut * a for a in a_t_dot]  # q = \dot{ϕ}/Mq


In [ ]:
plot(tlist, [ϕ_l[i][1] for i in 1:length(tlist)], label="ϕ₀(t) symplectic", xlabel="t", xlim=(0,20))
plot!(tlist, [q_l[i][1] for i in 1:length(tlist)], label="q₀(t) symplectic", xlabel="t")

In [ ]:
plot(tlist*κ, [ϕ_l[i][1] for i in 1:length(tlist)], label="", xlabel="t", xlim=(0,1))
plot!(t*κ, ϕ_analytic, label="", xlabel="κt")
title!("Comparison of the 1D chain and the CL master equation \n for a Lorentz-Drude spectral function")
#savefig("1D_chain_vs_CL_lorentz.pdf")

In [ ]:
F = fftshift(fft([ϕ_l[i][1] for i in 1:length(tlist)]))
fs = length(tlist)/(maximum(tlist)-minimum(tlist)) # sampling frequency
freqs = fftshift(fftfreq(length(tlist), fs))

# plots 
time_domain = plot(tlist, [ϕ_l[i][1] for i in 1:length(tlist)], title = "Signal", label='f')
freq_domain = plot(freqs, abs.(F), title = "Spectrum", xlim=(4.8, 5.2), label="abs.(F)") 
f_max=abs(freqs[argmax(abs.(F))])
vline!(freq_domain, [f_max], label = "ω₀")
plot(time_domain, freq_domain, layout = (2,1))

In [ ]:
f_max

In [ ]:
ω_sys/2pi-κ

In [ ]:
plot(av_ϕ_l_0)

In [ ]:
av_ϕ_l_0=averaging([ϕ_l[i][1] for i in 1:length(tlist)].^2,80)
plot(abs.(av_ϕ_l_0-av_ϕ_analitic)./av_ϕ_analitic, xlabel="κt", xlims=(0,300),yaxis=:log)
title!("Normalized error of the 1D chain \n and the CL master equation for a trapezoidal spectral density")

In [ ]:
function averaging(x::Vector{Float64}, window_size::Int)
    averaged = [mean(x[i*window_size+1:(i+1)*window_size]) for i in 0:div(length(x),window_size)-1]
    return averaged
end

# Check if all the eigenfrequencies of the modes are real

In [ ]:
using BlockDiagonals

In [ ]:

t=vcat(-system_chain_coupling*sqrt(Z_sys_prime)/2, hopping_energies)
D1=SymTridiagonal(ω, t)/2
D2 = zeros(n+1, n+1)
D2_begin=[0 -system_chain_coupling*sqrt(Z_sys_prime)/2
-system_chain_coupling*sqrt(Z_sys_prime)/2 0]/2
D2[1:2, 1:2] .= D2_begin

In [ ]:
D=[D1 D2
D2 D1]

In [ ]:
isposdef(D)

In [ ]:
eigvals(D)

# Non Linear Case

In [ ]:
using DifferentialEquations

## Ohmic bath

In [ ]:
n=500
onsite_energies,hopping_energies,system_chain_coupling = Ohmic_bath(n)

In [ ]:
# Definition of the first order ODE
function damped_non_linear_chain(du, u, p, t)
    cut=round(Int, length(u)/2)
    du[1] = ω_nl*u[cut+1]
    du[2:cut] = Mq_chain * u[cut+2:end]
    du[cut+1] = -E_J *sqrt(Z_nl)*sin(u[1] * sqrt(Z_nl)/ϕ_quantum) / ϕ_quantum + ω_nl* system_chain_coupling * sqrt(Z_nl) * u[2]
    du[cut+2:end] = - Mϕ_chain * u[2:cut]
    du[cut+2]+=system_chain_coupling * sqrt(Z_nl) * u[1]

end

function damped_non_linear_chain_2ODE(ddu,du,u,p,t)
    ddu[1] = - E_J*ω_nl * sqrt(Z_nl) * sin(u[1] * sqrt(Z_nl)/ϕ_quantum) / ϕ_quantum + ω_nl * system_chain_coupling * sqrt(Z_nl) * u[2]
    temp = Mϕ_chain*u[2:end]
    temp[1] -= system_chain_coupling * sqrt(Z_nl) * u[1]
    ddu[2:end]=-Mq_chain*temp[1:end]
    #ddu[1:end] = - Mq_bis*Mϕ_bis*u[1:end]
end

E_J = 90 *2pi /2  # ħ * GHz
ϕ_quantum = 1/2 # ħ/e 
ω_nl = sqrt(4*C_sys_inv*E_J)
Z_nl=C_sys_inv/ω_nl
gq=vcat(-hopping_energies)
gϕ=vcat(hopping_energies)
Mq_chain=SymTridiagonal(onsite_energies, gq)
Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)
Mq_bis=SymTridiagonal(vcat([ω_nl], onsite_energies), vcat([0],gq))
Mϕ_bis=SymTridiagonal(vcat([ω_nl], onsite_energies), vcat([-system_chain_coupling * sqrt(Z_nl)],gϕ));


ω = vcat(ω_sys_prime*0.7,onsite_energies) # GHz
g_q=vcat(0, -hopping_energies) 
g_ϕ=vcat(-system_chain_coupling*sqrt(Z_nl), hopping_energies) 

Mq=SymTridiagonal(ω, g_q)
Mϕ=SymTridiagonal(ω, g_ϕ);

In [ ]:
ϕ_init = 1.0
q_init = 1.0

ϕ_0 = vcat(ϕ_init/sqrt(Z_nl), fill(0.0, n))
q_0 = vcat(q_init * sqrt(Z_nl), fill(0.0, n))
ϕ_dot_0 = Mq_bis*q_0

X_0 = vcat(ϕ_0, q_0);


In [ ]:
tspan = (0.0, 1.0/κ) # Time span for the simulation
dt = 1 / (10 * maximum(vcat(ω_sys,onsite_energies))) # Time step for the simulation
prob = SecondOrderODEProblem(damped_non_linear_chain_2ODE, ϕ_dot_0, ϕ_0, tspan)
sol = solve(prob, VelocityVerlet(),dt=dt)
ϕ_dot = [getindex.(sol.u,i) for i in 1:n+1]
ϕ = [getindex.(sol.u,i+n+1) for i in 1:n+1]

In [ ]:

inv_Mq_bis = inv(Mq_bis)
q = [inv_Mq_bis*getindex.(ϕ_dot,t) for t in 1:length(sol.t)]
q = [getindex.(q,i) for i in 1:n+1]

E=q[1].^2 * ω_nl /2 - E_J*cos.(ϕ[1]*sqrt(Z_nl)/ϕ_quantum) - system_chain_coupling *sqrt(Z_nl) * ϕ[1] .* ϕ[2] + 1/2*[getindex.(q,i)[2:end]'*Mq_chain*getindex.(q,i)[2:end] + getindex.(ϕ,i)[2:end]'*Mϕ_chain*getindex.(ϕ,i)[2:end] for i in 1:length(sol.t)];
#E=1/2*[getindex.(q,i)[1:end]'*Mq_bis*getindex.(q,i)[1:end] + getindex.(ϕ,i)[1:end]'*Mϕ_bis*getindex.(ϕ,i)[1:end] for i in 1:length(sol.t)];

In [ ]:
plot(E)

In [ ]:
plot(ϕ[1],q[1])

In [ ]:
plot(sol.t,ϕ[1])

In [ ]:
F = fftshift(fft(ϕ[1]))
fs = length(sol.t)/(maximum(sol.t)-minimum(sol.t)) # sampling frequency
freqs = fftshift(fftfreq(length(sol.t), fs))

# plots 
time_domain = plot(sol.t,ϕ[1], title = "Signal", label="")
freq_domain = plot(freqs, abs.(F), title = "Spectrum", xlim=(3, 6), label="") 
f_max=abs(freqs[argmax(abs.(F))])
vline!(freq_domain, [f_max], label = "")
plot(time_domain, freq_domain, layout = (2,1))

In [ ]:
f_max

In [ ]:
ω_nl/2pi